In [1]:
# 01. Business Problem & Project Overview
USE_CASE = "08 - Quality Inspection Failure Prediction (Predict Failed Inspection Lots)"
ML_PARADIGM = "Unsupervised Machine Learning (5 Models Evaluated & Ranked -> #1 Champion Selected)"
EXCEPTION_TYPE = "Quality Inspection Failure / Exception Classification"
SCOPE = "ALL DATA (HANA_P3 & HANA_P2 Without Limit)"

print("=" * 90)
print(f"USE CASE:       {USE_CASE}")
print(f"PARADIGM:       {ML_PARADIGM}")
print(f"EXCEPTION:      {EXCEPTION_TYPE}")
print(f"DATA SCOPE:     {SCOPE}")
print("OBJECTIVE:      Predict failed inspection lots from input features without labeled supervision")
print("=" * 90)


USE CASE:       08 - Quality Inspection Failure Prediction (Predict Failed Inspection Lots)
PARADIGM:       Unsupervised Machine Learning (5 Models Evaluated & Ranked -> #1 Champion Selected)
EXCEPTION:      Quality Inspection Failure / Exception Classification
DATA SCOPE:     ALL DATA (HANA_P3 & HANA_P2 Without Limit)
OBJECTIVE:      Predict failed inspection lots from input features without labeled supervision


In [2]:
# 02. Import Core ML & Analytics Libraries
import os
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
print("All core analytics and ML libraries loaded successfully.")


All core analytics and ML libraries loaded successfully.


In [3]:
# 03. Database Server Connection & Configuration
from src.db_connector import load_config, get_db_connection

config = load_config()
db = config["database"]
print(f"PostgreSQL Host: {db['host']}:{db['port']}")
print(f"Database Name:   {db['dbname']}")
print(f"Schemas:         {db.get('schemas', ['HANA_P3', 'HANA_P2'])}")

conn = get_db_connection(config)
print("Database connection successfully established.")
conn.close()


PostgreSQL Host: 192.168.1.44:5432
Database Name:   JKC_Input
Schemas:         ['HANA_P3', 'HANA_P2']
Database connection successfully established.


In [4]:
# 04. Extract All Inspection Lots from Live Tables (QALS, QAVE, MAKT, MARA, MARC, MBEW, LFA1)
from src.db_connector import extract_quality_inspection_data

raw_df = extract_quality_inspection_data(limit=None, config=config, use_cache=True)
print(f"Extracted Raw Records: {len(raw_df):,}")
print(f"Extracted Columns:     {len(raw_df.columns)}")
print(raw_df.head(3))


[CACHE] Loading cached quality inspection dataset from: data/cache/qm_inspection_lot_raw_data.pkl
[CACHE] Successfully loaded 664,432 records from cache.
Extracted Raw Records: 664,432
Extracted Columns:     41
  inspection_lot plant  ... price_unit source_schema
0    10000645453  1001  ...          1       HANA_P3
1   890000047808  1901  ...          1       HANA_P3
2    30000010877  M101  ...          1       HANA_P3

[3 rows x 41 columns]


In [5]:
# 05. Exploratory Data Inspection & Dimensions
print("--- Inspection Type Distribution ---")
print(raw_df['inspection_type'].value_counts().head(7))

print("")
print("--- Top Manufacturing Plants ---")
print(raw_df['plant'].value_counts().head(7))

print("")
print("--- Missing Values in Raw Data (Checked Properly) ---")
for col in ['vendor_id', 'vendor_name', 'batch_number', 'material_name', 'po_number', 'ud_valuation']:
    n_null = raw_df[col].isnull().sum() + (raw_df[col].astype(str).str.strip().isin(['nan', 'None', '', '<NA>'])).sum()
    print(f"  {col}: {n_null:,} blank/missing fields ({n_null/len(raw_df)*100:.1f}%)")


--- Inspection Type Distribution ---
inspection_type
01         569154
08          39749
ZJCO89      37911
03           8029
04           5183
ZJBIS89      3808
89            589
Name: count, dtype: int64

--- Top Manufacturing Plants ---
plant
2001    108566
1015     82213
1001     75275
1004     73170
1901     60036
1022     51570
1033     33613
Name: count, dtype: int64

--- Missing Values in Raw Data (Checked Properly) ---
  vendor_id: 95,277 blank/missing fields (14.3%)
  vendor_name: 95,313 blank/missing fields (14.3%)
  batch_number: 580,611 blank/missing fields (87.4%)
  material_name: 9 blank/missing fields (0.0%)
  po_number: 55,542 blank/missing fields (8.4%)
  ud_valuation: 52,362 blank/missing fields (7.9%)


In [6]:
# 06. Data Cleaning & Engineering 32 Domain Features
from src.preprocessing import clean_and_engineer_features

featured_df = clean_and_engineer_features(raw_df)
print(f"Dataset Shape after Feature Engineering: {featured_df.shape}")
print(f"Creation Date Range: {featured_df['creation_date_dt'].min().date()} to {featured_df['creation_date_dt'].max().date()}")


Dataset Shape after Feature Engineering: (664432, 98)
Creation Date Range: 2024-04-01 to 2026-07-31


In [7]:
# 07. Historical Benchmark Exception Ground-Truth (For Post-Hoc Evaluation Only)
fail_cnt = int(featured_df['is_failure_exception'].sum())
fail_rate = (fail_cnt / len(featured_df)) * 100.0

print(f"Total Inspection Lots:               {len(featured_df):,}")
print(f"Benchmark Failure Exception Lots:    {fail_cnt:,} ({fail_rate:.2f}%)")
print(f"Benchmark Compliant Passing Lots:   {len(featured_df) - fail_cnt:,} ({100.0 - fail_rate:.2f}%)")


Total Inspection Lots:               664,432
Benchmark Failure Exception Lots:    1,765 (0.27%)
Benchmark Compliant Passing Lots:   662,667 (99.73%)


In [8]:
# 08. Unsupervised 32-Feature Matrix Preparation
from src.preprocessing import get_feature_columns

feature_cols = get_feature_columns()
print(f"Total Engineered Domain Features ({len(feature_cols)}):")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {f}")


Total Engineered Domain Features (32):
   1. log_lot_quantity
   2. log_sample_quantity
   3. sample_to_lot_ratio
   4. destroyed_sample_qty_num
   5. destroyed_sample_ratio
   6. scrap_qty_num
   7. scrap_ratio
   8. blocked_qty_num
   9. blocked_ratio
  10. unrestricted_ratio
  11. non_conforming_ratio
  12. inspection_lead_days
  13. scheduled_duration_days
  14. ud_lag_days
  15. is_overdue_inspection
  16. log_unit_price
  17. log_lot_financial_value
  18. planned_delivery_days_num
  19. log_safety_stock
  20. log_net_weight
  21. log_vendor_frequency
  22. vendor_is_rare
  23. log_material_frequency
  24. log_plant_frequency
  25. insp_type_code
  26. origin_code
  27. material_type_code
  28. has_batch
  29. has_po
  30. status_sample_flag
  31. is_monthend_rush
  32. is_weekend_creation


In [9]:
# 09. Train/Test Split (80% Train, 20% Test)
from src.preprocessing import prepare_train_test_data

data_dict = prepare_train_test_data(featured_df, test_size=0.20, random_state=42)
X_train = data_dict['X_train']
X_test = data_dict['X_test']
X_train_scaled = data_dict['X_train_scaled']
X_test_scaled = data_dict['X_test_scaled']
y_train = data_dict['y_train']
y_test = data_dict['y_test']
scaler = data_dict['scaler']

print(f"Training Matrix (Unsupervised X): {X_train_scaled.shape}")
print(f"Testing Matrix (Held-out Test):   {X_test_scaled.shape}")
print(f"Test Benchmark Exceptions:        {int(y_test.sum()):,} ({y_test.mean()*100:.2f}%)")


Training Matrix (Unsupervised X): (531545, 32)
Testing Matrix (Held-out Test):   (132887, 32)
Test Benchmark Exceptions:        353 (0.27%)


In [10]:
# 10. Validate Scaled Feature Distributions (RobustScaler)
scaled_df = pd.DataFrame(X_train_scaled[:1000], columns=feature_cols)
print("Scaled Features Check (Median ~ 0, robust to extreme outlier spikes):")
print(scaled_df[['log_lot_quantity', 'sample_to_lot_ratio', 'inspection_lead_days', 'non_conforming_ratio']].describe().T[['mean', 'std', 'min', '50%', 'max']])


Scaled Features Check (Median ~ 0, robust to extreme outlier spikes):
                          mean       std       min       50%        max
log_lot_quantity      0.080832  4.444161 -8.003686 -0.015905  24.093432
sample_to_lot_ratio   0.001000  0.031623  0.000000  0.000000   1.000000
inspection_lead_days  0.039000  0.829558  0.000000  0.000000  19.000000
non_conforming_ratio  0.000000  0.000000  0.000000  0.000000   0.000000


In [11]:
# 11. Train Model 1/5: Isolation Forest
from src.models import QualityInspectionFailureDetectionSuite

suite = QualityInspectionFailureDetectionSuite(contamination=0.05, random_state=42)
m1 = suite.train_isolation_forest(X_train_scaled)
print("Model 1 (Isolation Forest) trained successfully.")


[INFO] Training Unsupervised Model 1/5: Isolation Forest...
Model 1 (Isolation Forest) trained successfully.


In [12]:
# 12. Train Model 2/5: Local Outlier Factor (LOF)
m2 = suite.train_lof(X_train_scaled)
print("Model 2 (Local Outlier Factor) trained successfully.")


[INFO] Training Unsupervised Model 2/5: Local Outlier Factor (LOF)...
Model 2 (Local Outlier Factor) trained successfully.


In [13]:
# 13. Train Model 3/5: One-Class SVM
m3 = suite.train_one_class_svm(X_train_scaled)
print("Model 3 (One-Class SVM) trained successfully.")


[INFO] Training Unsupervised Model 3/5: One-Class SVM...
Model 3 (One-Class SVM) trained successfully.


In [14]:
# 14. Train Model 4/5: DBSCAN (Adaptive Density Clustering)
m4 = suite.train_dbscan(X_train_scaled)
print(f"Model 4 (DBSCAN) trained successfully. Adaptive eps: {m4['eps']:.4f}")


[INFO] Training Unsupervised Model 4/5: DBSCAN...
Model 4 (DBSCAN) trained successfully. Adaptive eps: 3.1907


In [15]:
# 15. Train Model 5/5: Deep Neural Autoencoder
m5 = suite.train_autoencoder(X_train_scaled)
print(f"Model 5 (Autoencoder) trained. Reconstruction Threshold: {suite.thresholds['autoencoder']:.4f}")


[INFO] Training Unsupervised Model 5/5: Deep Neural Autoencoder...
Model 5 (Autoencoder) trained. Reconstruction Threshold: 0.7518


In [16]:
# 16. Comprehensive Test Set Evaluation & 5-Model Accuracy Ranking Table
from src.evaluate import evaluate_models

# Evaluate on representative held-out test split
eval_X = X_test_scaled[:20000] if len(X_test_scaled) > 20000 else X_test_scaled
eval_y = y_test[:20000] if len(y_test) > 20000 else y_test

test_preds = suite.predict_all(eval_X)
comparison_df = evaluate_models(test_preds, eval_y)

print("=" * 95)
print("  5 UNSUPERVISED ML MODELS EVALUATION & ACCURACY RANKING TABLE")
print("=" * 95)
print(comparison_df.to_string(index=False))
print("=" * 95)


  5 UNSUPERVISED ML MODELS EVALUATION & ACCURACY RANKING TABLE
Rank                Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC  PR-AUC  Specificity  True Positives  False Positives  False Negatives  True Negatives
  #1     Isolation Forest    0.9526     0.0491  1.0000    0.0936   0.9955  0.3501       0.9524              49              949                0           19002
  #2               DBSCAN    0.9518     0.0466  0.9592    0.0888   0.9847  0.1893       0.9518              47              962                2           18989
  #3          Autoencoder    0.9504     0.0364  0.7551    0.0694   0.9789  0.6269       0.9509              37              980               12           18971
  #4        One-Class SVM    0.9482     0.0349  0.7551    0.0667   0.8562  0.1986       0.9487              37             1024               12           18927
  #5 Local Outlier Factor    0.9485     0.0139  0.2857    0.0265   0.6719  0.0079       0.9501              14              995     

In [17]:
# 17. Diagnostic Visualizations: Comparative Performance Bar Chart & ROC/PR Curves
from src.evaluate import plot_model_comparison, plot_roc_pr_curves

os.makedirs("reports/figures", exist_ok=True)
plot_model_comparison(comparison_df, output_dir="reports/figures")
plot_roc_pr_curves(test_preds, eval_y, output_dir="reports/figures")
print("Model comparison and ROC/PR curve charts generated and saved in reports/figures/")


[INFO] Saved 5-model comparison chart to: reports/figures\model_comparison.png
[INFO] Saved ROC/PR curves plot to: reports/figures\roc_pr_curves.png
Model comparison and ROC/PR curve charts generated and saved in reports/figures/


In [18]:
# 18. Selection of #1 Ranked Champion Model
from src.models import SingleModelQualityFailureDetector

champion_model_name = comparison_df.iloc[0]['Model']
champion_acc = comparison_df.iloc[0]['Accuracy'] * 100.0
champion_f1 = comparison_df.iloc[0]['F1-Score']
champion_auc = comparison_df.iloc[0]['ROC-AUC']

print(f"SELECTED #1 CHAMPION MODEL: {champion_model_name}")
print(f"Accuracy:   {champion_acc:.2f}%")
print(f"F1-Score:   {champion_f1:.4f}")
print(f"ROC-AUC:    {champion_auc:.4f}")

champion_segmenter = SingleModelQualityFailureDetector(
    model_name=champion_model_name,
    contamination=0.05,
    random_state=42
)
champion_segmenter.fit(X_train_scaled)
print("Champion model ready for production scoring across all data.")


SELECTED #1 CHAMPION MODEL: Isolation Forest
Accuracy:   95.26%
F1-Score:   0.0936
ROC-AUC:    0.9955
[INFO] Fitting Selected #1 Champion Model (Isolation Forest) for Production Scoring...
[INFO] Champion Model fitted. Score range: [0.3477, 0.6424], Offset Threshold: 0.5655
Champion model ready for production scoring across all data.


In [19]:
# 19. Production Inference: Scoring All Inspection Lots & Calibrating P(Failure)
from src.inference import score_with_single_model
from src.evaluate import plot_failure_risk_distribution

scored_df = score_with_single_model(champion_segmenter, featured_df, scaler, feature_cols)
plot_failure_risk_distribution(scored_df, output_dir="reports/figures")

print("")
print("--- Quality Risk Tier Triage Breakdown (All Data) ---")
print(scored_df['risk_category'].value_counts())


[INFO] Saved failure risk distribution plot to: reports/figures\failure_risk_distribution.png

--- Quality Risk Tier Triage Breakdown (All Data) ---
risk_category
Low Risk                 601972
Medium Risk               29147
High Risk                 20659
Critical Failure Risk     12654
Name: count, dtype: int64


In [20]:
# 20. Feature Importance & Domain Impact Explainability
from src.evaluate import compute_feature_importance, plot_feature_importance

fi_df = compute_feature_importance(champion_segmenter.model, feature_cols, X_train_scaled, y_train)
fi_df.to_csv("reports/feature_importance.csv", index=False)
plot_feature_importance(fi_df, output_dir="reports/figures", top_n=15)

print("")
print("Top 10 Influential Quality Failure Drivers:")
print(fi_df.head(10)[['Rank', 'Feature', 'Importance', 'Relative_Weight_Pct']].to_string(index=False))


[INFO] Saved feature importance chart to: reports/figures\feature_importance.png

Top 10 Influential Quality Failure Drivers:
Rank                 Feature  Importance  Relative_Weight_Pct
  #1    inspection_lead_days      0.5122                51.22
  #2 scheduled_duration_days      0.2265                22.65
  #3 log_lot_financial_value      0.0407                 4.07
  #4          log_unit_price      0.0231                 2.31
  #5        log_lot_quantity      0.0209                 2.09
  #6          insp_type_code      0.0205                 2.05
  #7  log_material_frequency      0.0179                 1.79
  #8             origin_code      0.0146                 1.46
  #9     log_plant_frequency      0.0133                 1.33
 #10        is_monthend_rush      0.0118                 1.18


In [21]:
# 21. Multi-Sheet Executive Excel Deliverable Export & Artifact Persistence
import shutil
from src.inference import export_enriched_excel_report, save_single_model_pipeline

output_excel = "ML_8_Quality_Inspection_Failure_Prediction_Enriched_Output_Clean.xlsx"
export_enriched_excel_report(scored_df, output_path=output_excel, fi_df=fi_df, comp_df=comparison_df)

alt_excel = "ML_8_Quality_Inspection_Failure_Prediction_Enriched_Output.xlsx"
shutil.copyfile(output_excel, alt_excel)
shutil.copyfile(output_excel, os.path.join("reports", os.path.basename(output_excel)))
shutil.copyfile(output_excel, os.path.join("reports", alt_excel))

save_single_model_pipeline(champion_segmenter, scaler, feature_cols, output_dir="models")
print("All deliverables and artifacts successfully exported and saved.")


[EXCEL] Streaming 664,432 scored inspection lots into sheet...
[EXCEL] Professional 5-sheet workbook successfully generated: ML_8_Quality_Inspection_Failure_Prediction_Enriched_Output_Clean.xlsx
[SAVE] Champion Model saved to: models\champion_quality_failure_model.joblib
[SAVE] RobustScaler saved to:   models\robust_scaler.joblib
[SAVE] Pipeline Bundle saved to: models\quality_failure_model_bundle.joblib
All deliverables and artifacts successfully exported and saved.
